In [0]:
import pandas as pd
import numpy as np

In [0]:
temporal_file = "/Volumes/workspace/default/ucf_crime/annotations/temporal/Temporal_Anomaly_Annotation_for_Testing_Videos.txt"


def load_temporal_annotations(temporal_file):
    columns = ["video", "class", "start_1", "end_1", "start_2", "end_2"]
    df = pd.read_csv(temporal_file, sep=r"\s+", names=columns)
    return df

temporal_df = load_temporal_annotations(temporal_file)
temporal_df

In [0]:
print("data types:")
print(temporal_df.dtypes)

display(
    temporal_df["class"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)

In [0]:
has_second_event = (temporal_df["start_2"] != -1) & (temporal_df["end_2"] != -1)
print("videos with second event: ", has_second_event.sum())
print("videos without second event: ", (~has_second_event).sum())


In [0]:
def clean_temporal_annotations(temporal_df):
    """
    - removes videos with no second event
    - removes videos with second event that is before the first event
    """
    rows = []
    for _, row in df.iterrows():
        # normal video: keep one row with no anomaly interval
        if row["class"] == "Normal":
            rows.append({
                "video": row["video"],
                "class": row["class"],
                "interval_id": 0,
                "start_frame": None,
                "end_frame": None
            })
            continue
        # first anomaly interval
        if row["start_1"] != -1 and row["end_1"] != -1:
            rows.append({
                "video": row["video"],
                "class": row["class"],
                "interval_id": 1,
                "start_frame": row["start_1"],
                "end_frame": row["end_1"]
            })
        # second anomaly interval
        if row["start_2"] != -1 and row["end_2"] != -1:
            rows.append({
                "video": row["video"],
                "class": row["class"],
                "interval_id": 2,
                "start_frame": row["start_2"],
                "end_frame": row["end_2"]
            })
    return pd.DataFrame(rows)

temporal_clean = clean_temporal_annotations(temporal_df)
display(temporal_clean)

In [0]:
temporal_clean = clean_temporal_annotations(temporal_df)
print("Rows:", len(temporal_clean))
print("Unique videos:", temporal_clean["video"].nunique())

print("\nMissing values:")
print(temporal_clean.isna().sum())

In [0]:
output_path = (
    "/Volumes/workspace/default/ucf_crime/"
    "annotations/temporal/temporal_clean.csv"
)
temporal_clean.to_csv(output_path, index=False)